<a href="https://colab.research.google.com/github/NazHub1993/ML_Notebooks/blob/main/Real_State_Project_Codebasics.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1388]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib


In [1389]:
df=pd.read_csv('bengaluru_house_prices.csv')
df.head()

,area_type,availability,location,size,society,total_sqft,bath,balcony,price
0,Super built-up Area,19-Dec,Electronic City Phase II,2 BHK,Coomee,1056,2.0,1.0,39.07
1,Plot Area,Ready To Move,Chikka Tirupathi,4 Bedroom,Theanmp,2600,5.0,3.0,120.00
2,Built-up Area,Ready To Move,Uttarahalli,3 BHK,NaN,1440,2.0,3.0,62.00
3,Super built-up Area,Ready To Move,Lingadheeranahalli,3 BHK,Soiewre,1521,3.0,1.0,95.00
4,Super built-up Area,Ready To Move,Kothanur,2 BHK,NaN,1200,2.0,1.0,51.00


In [1390]:
df.shape

(13320, 9)

In [1391]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 13320 entries, 0 to 13319
Data columns (total 9 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   area_type     13320 non-null  object 
 1   availability  13320 non-null  object 
 2   location      13319 non-null  object 
 3   size          13304 non-null  object 
 4   society       7818 non-null   object 
 5   total_sqft    13320 non-null  object 
 6   bath          13247 non-null  float64
 7   balcony       12711 non-null  float64
 8   price         13320 non-null  float64
dtypes: float64(3), object(6)
memory usage: 936.7+ KB


In [1392]:
df['area_type'].unique()

array(['Super built-up  Area', 'Plot  Area', 'Built-up  Area',
       'Carpet  Area'], dtype=object)

In [1393]:
df['area_type'].value_counts()

,count
area_type,
Super built-up Area,8790
Built-up Area,2418
Plot Area,2025
Carpet Area,87


In [1394]:
df.shape

(13320, 9)

#Dropping the features that are unnecessary in predicting the price

In [1395]:
df2 = df.drop(['area_type','society','balcony','availability'],axis='columns')
df2.shape

(13320, 5)

#Handling the null values

In [1396]:
df2.isnull().sum()

,0
location,1
size,16
total_sqft,0
bath,73
price,0


#After dropping out rows with missing values

In [1397]:
df3=df2.dropna()
df3.isnull().sum()

,0
location,0
size,0
total_sqft,0
bath,0
price,0


#Feature Engineering-Part1

#I have converted size column to BHK column which now includes How many bedrooms do I have in the flat
#I have converted total_sqft column values that have ranges like [1150-1250] to an average float value




In [1398]:
df3.location.unique()

array(['Electronic City Phase II', 'Chikka Tirupathi', 'Uttarahalli', ...,
       '12th cross srinivas nagar banshankari 3rd stage',
       'Havanur extension', 'Abshot Layout'], dtype=object)

In [1399]:
df3['size'].unique()

array(['2 BHK', '4 Bedroom', '3 BHK', '4 BHK', '6 Bedroom', '3 Bedroom',
       '1 BHK', '1 RK', '1 Bedroom', '8 Bedroom', '2 Bedroom',
       '7 Bedroom', '5 BHK', '7 BHK', '6 BHK', '5 Bedroom', '11 BHK',
       '9 BHK', '9 Bedroom', '27 BHK', '10 Bedroom', '11 Bedroom',
       '10 BHK', '19 BHK', '16 BHK', '43 Bedroom', '14 BHK', '8 BHK',
       '12 Bedroom', '13 BHK', '18 Bedroom'], dtype=object)

#You can see the values are in string .
#Again you can see the strings are not also in same format
one is saying BHK another is saying bedroom.
# My requirement is taking only the numeric value that is the first token for every string and save it inot a new column

In [1400]:
df3['BHK']=df3['size'].apply(lambda x:int(x.split(" ")[0]))
df3['BHK'].unique()

/tmp/ipykernel_843/566543680.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df3['BHK']=df3['size'].apply(lambda x:int(x.split(" ")[0]))


array([ 2,  4,  3,  6,  1,  8,  7,  5, 11,  9, 27, 10, 19, 16, 43, 14, 12,
       13, 18])

In [1401]:
df4=df3.drop('size',axis=1)
df4.head()

,location,total_sqft,bath,price,BHK
0,Electronic City Phase II,1056,2.0,39.07,2
1,Chikka Tirupathi,2600,5.0,120.00,4
2,Uttarahalli,1440,2.0,62.00,3
3,Lingadheeranahalli,1521,3.0,95.00,3
4,Kothanur,1200,2.0,51.00,2


In [1402]:
df4['total_sqft'].unique()

array(['1056', '2600', '1440', ..., '1133 - 1384', '774', '4689'],
      dtype=object)

Above shows that total_sqft can be a range (e.g. 2100-2850). For such case we can just take average of min and max value in the range. There are other cases such as 34.46Sq. Meter which one can convert to square ft using unit conversion. I am going to just drop such corner cases to keep things simple

In [1403]:
df4['total_sqft']

,total_sqft
0,1056
1,2600
2,1440
3,1521
4,1200
...,...
13315,3453
13316,3600
13317,1141
13318,4689


In [1404]:
def convert_sqft_to_num(x):
  if isinstance(x,str) and '-' in x:
      tokens=x.split('-')
      try:
        return (float(tokens[0])+float(tokens[1]))/2
      except:
        return None
  else:
    try:
      return float(x)
    except:
      return None


In [1405]:
df4['total_sqft']=df4['total_sqft'].apply(convert_sqft_to_num)
df4['total_sqft'].unique()

array([1056. , 2600. , 1440. , ..., 1258.5,  774. , 4689. ])

In [1406]:
df4['total_sqft'].dtype

dtype('float64')

In [1407]:
df.loc[410]

,410
area_type,Super built-up Area
availability,Ready To Move
location,Kengeri
size,1 BHK
society,NaN
total_sqft,34.46Sq. Meter
bath,1.0
balcony,0.0
price,18.5


In [1408]:
df4['total_sqft'].isnull().sum()

np.int64(46)

In [1409]:
df4=df4.dropna()

In [1410]:
df4.isnull().sum()

,0
location,0
total_sqft,0
bath,0
price,0
BHK,0


In [1411]:
df5=df4.copy()


#Feature Engineering - Part 2
#I have created another column named price_per_sqft here.
#Since location is a categorical column and too many locations would ruin the data by increasing dimensionaltiy
so the loactions which have occurences less than 10 I will define them as other.   

In [1412]:
df5=df4.copy()
df5['price_per_sqft'] = (df5['price'] * 100000) / df5['total_sqft']
df5.head()

,location,total_sqft,bath,price,BHK,price_per_sqft
0,Electronic City Phase II,1056.0,2.0,39.07,2,3699.810606
1,Chikka Tirupathi,2600.0,5.0,120.00,4,4615.384615
2,Uttarahalli,1440.0,2.0,62.00,3,4305.555556
3,Lingadheeranahalli,1521.0,3.0,95.00,3,6245.890861
4,Kothanur,1200.0,2.0,51.00,2,4250.000000


In [1413]:
location_stats=df5['location'].value_counts(ascending=False)

In [1414]:
print(location_stats)

location
Whitefield                         532
Sarjapur  Road                     392
Electronic City                    302
Kanakpura Road                     264
Thanisandra                        232
                                  ... 
beml layout, basaveshwara nagar      1
Sadhguru Layout                      1
Chikbasavanapura                     1
Electronic City Phase 1,             1
Chuchangatta Colony                  1
Name: count, Length: 1298, dtype: int64


In [1415]:
location_stats_less_than_10=location_stats[location_stats<=10]
location_stats_less_than_10

,count
location,
BTM 1st Stage,10
Basapura,10
Gunjur Palya,10
Naganathapura,10
Ganga Nagar,10
...,...
"beml layout, basaveshwara nagar",1
Sadhguru Layout,1
Chikbasavanapura,1


In [1416]:
len(location_stats_less_than_10)

1058

In [1417]:
len(location_stats)

1298

In [1418]:
df5['location']=df5['location'].apply(lambda x: 'other' if x in location_stats_less_than_10 else x)
df5['location'].nunique()

241

#I have only 241 different locations now which can be easily processed through one hot encoding

#Now what I want is for each location I want to consider the points within one standard deviation. And the rest of them I want to remove as outliers

In [1419]:
df5['price_per_sqft'].describe()

,price_per_sqft
count,1.320000e+04
mean,7.920759e+03
std,1.067272e+05
min,2.678298e+02
25%,4.267701e+03
50%,5.438331e+03
75%,7.317073e+03
max,1.200000e+07


In [1420]:
df5 = df5[~(df5.total_sqft/df5.BHK<300)]
df5.shape

(12456, 6)

In [1421]:
def remove_outliers(df):
  df_output=pd.DataFrame()
  for key,subdf in df.groupby('location'):
    mean_=np.mean(subdf.price_per_sqft)
    std_deviation=np.std(subdf.price_per_sqft)

    gen_df=subdf[(subdf.price_per_sqft>(mean_-std_deviation)) & (subdf.price_per_sqft<=(mean_+std_deviation))]
    df_output=pd.concat([df_output,gen_df],ignore_index=True)

  return df_output


df5=remove_outliers(df5)


In [1422]:
df5.shape

(10245, 6)

#After removing outliers from the data

In [1423]:
df5['price_per_sqft'].describe()

,price_per_sqft
count,10245.000000
mean,5657.835532
std,2266.165749
min,1250.000000
25%,4244.762955
50%,5173.279759
75%,6426.099852
max,24509.803922


#Now for the same location price for 2BHK can't be more than 3BHK

In [1424]:
def remove_bhk_outliers(df):
    exclude_indices = np.array([])
    for location, location_df in df.groupby('location'):
        bhk_stats = {}
        for bhk, bhk_df in location_df.groupby('BHK'):
            bhk_stats[bhk] = {
                'mean': np.mean(bhk_df.price_per_sqft),
                'std': np.std(bhk_df.price_per_sqft),
                'count': bhk_df.shape[0]
            }
        for bhk, bhk_df in location_df.groupby('BHK'):
            stats = bhk_stats.get(bhk-1)
            if stats and stats['count']>5:
                exclude_indices = np.append(exclude_indices, bhk_df[bhk_df.price_per_sqft<(stats['mean'])].index.values)
    return df.drop(exclude_indices,axis='index')
df6 = remove_bhk_outliers(df5)
# df8 = df7.copy()
df6.shape



(7331, 6)

In [1425]:
df5.BHK.describe()

,BHK
count,10245.000000
mean,2.572279
std,0.897398
min,1.000000
25%,2.000000
50%,2.000000
75%,3.000000
max,16.000000


In [1426]:
df5.shape

(10245, 6)

In [1427]:
df6.BHK.describe()

,BHK
count,7331.000000
mean,2.498704
std,0.926106
min,1.000000
25%,2.000000
50%,2.000000
75%,3.000000
max,16.000000


In [1428]:
df6.shape

(7331, 6)

In [1429]:
df6=df6[df6.bath<df6.BHK+2]

In [1430]:
df6.shape

(7253, 6)

#So rows are reduced!!!

In [1431]:
df6.drop(columns=['price_per_sqft'],inplace=True)
df6.head()

,location,total_sqft,bath,price,BHK
1,Devarachikkanahalli,1250.0,2.0,40.0,2
2,Devarachikkanahalli,1200.0,2.0,83.0,2
3,Devarachikkanahalli,1170.0,2.0,40.0,2
4,Devarachikkanahalli,1425.0,2.0,65.0,3
5,Devarachikkanahalli,947.0,2.0,43.0,2


In [1432]:
df6.to_csv('cleaned_data.csv')

In [1433]:
df6.shape

(7253, 5)

In [1434]:
data=df6.copy()

In [1435]:
data.shape

(7253, 5)

In [1436]:
X=data.drop(columns='price',axis='columns')
Y=data.price

#!!!Time to create the model

In [1437]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression,Lasso,Ridge
from sklearn.preprocessing import OneHotEncoder,StandardScaler
from sklearn.compose import make_column_transformer
from sklearn.pipeline import make_pipeline
from sklearn.metrics import r2_score

In [1438]:
x_train,x_test,y_train,y_test=train_test_split(X,Y,test_size=0.2,random_state=42)

In [1439]:
col_trans=make_column_transformer(
    (OneHotEncoder(sparse_output=False),['location'])
    ,remainder='passthrough')

In [1440]:
scaler=StandardScaler()
lr=LinearRegression()

In [1441]:
pipe=make_pipeline(col_trans,scaler,lr)


In [1442]:
pipe.fit(x_train,y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('columntransformer',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('onehotencoder',
                                                  OneHotEncoder(sparse_output=False),
                                                  ['location'])])),
                ('standardscaler', StandardScaler()),
                ('linearregression', LinearRegression())])

In [1443]:
y_pred_lr=pipe.predict(x_test)
r2_score(y_test,y_pred_lr)

0.8595687461071945

In [1444]:
ls=Lasso()
pipe=make_pipeline(col_trans,scaler,ls)
pipe.fit(x_train,y_train)
y_pred_lasso=pipe.predict(x_test)
r2_score(y_test,y_pred_lasso)

0.8491777134596019

In [1445]:
ridge=Ridge()
pipe=make_pipeline(col_trans,scaler,ridge)
pipe.fit(x_train,y_train)
y_pred_ridge=pipe.predict(x_test)
r2_score(y_test,y_pred_ridge)

0.8595587415202107

In [1446]:
import pickle
pickle.dump(pipe,open('RidgeModel_Housing_Project.pkl','wb'))